In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Water Crisis Management Simulation

In [ ]:
import os
import sys
import random
import math
import subprocess
import importlib
from typing import Dict, List, Set, Any, Optional
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

# 1. Path Configuration
root_dir = '/content/drive/MyDrive/Data Science/Honor_Program_private/progetto_locale'
dyn_wntr_path = os.path.join(root_dir, 'Dyn-WNTR')
lorasim_path = os.path.join(root_dir, 'LoRaSim-master/LoRaSim')

if os.path.exists(root_dir):
    for path in [root_dir, dyn_wntr_path, lorasim_path]:
        if os.path.exists(path) and path not in sys.path:
            sys.path.insert(0, path)
    print(f" Directory configured: {root_dir}")
    print(f" Path Dyn-WNTR: {dyn_wntr_path}")
    print(f" Path LoRaSim: {lorasim_path}")
else:
    print(f" Warning: The path {root_dir} does not exist.")

# 2. Native Component Compilation (C++)
print(" Compiling native components (C++)...")
try:
    result = subprocess.run(
        ["python", "setup.py", "build_ext", "--inplace"],
        cwd=dyn_wntr_path,
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(" Compilation completed successfully.")
    else:
        print(" Error during compilation:")
        print(result.stderr)
except Exception as e:
    print(f" Critical error during build process: {e}")

# 3. Module Import and Reload
try:
    import mwntr
    importlib.reload(mwntr)
    from mwntr.network import LinkStatus
    from mwntr.sim.interactive_network_simulator import MWNTRInteractiveSimulator
    print(" Module mwntr imported correctly.")
except ImportError as e:
    print(f" Error importing mwntr: {e}")

## 1. LoRaSystem (Custom LoRaSim Integration)
This class manages the LoRaWAN communication layer using Markov Chain models for packet loss simulation.

In [ ]:
class LoRaSystem:
    """Simulates LoRaWAN communication using Markov models with realistic transmission timing."""
    def __init__(self):
        self.sensors = {}
        self.total_transmissions = 0
        self.total_collisions = 0
        self.tx_interval_s = 3600
        self.models_dir = os.path.join(lorasim_path, 'Models')

    def _get_best_model(self, distance_km: float, sf: int):
        from LoRaSim.MarkovChain import MarkovChain
        dist_str = "650m" if distance_km <= 1.0 else "2km"
        dr_str = f"DR{max(0, min(6, 12 - sf))}"
        available = [f for f in os.listdir(self.models_dir) if f.endswith('.ini')]
        best = next((m for m in available if dist_str in m and dr_str in m), available[0])
        model = MarkovChain()
        model.loadFromFile(os.path.join(self.models_dir, best))
        return model

    def register_sensor(self, sensor_id, distance_km=1.0, sf=None):
        if sf is None: sf = random.choice([7, 8, 9, 10, 11, 12])
        self.sensors[sensor_id] = {
            'sf': sf, 'distance': distance_km, 'data': {},
            'model': self._get_best_model(distance_km, sf),
            'state': 1, 'last_tx_time': -9999.0
        }
        print(f" Registered {sensor_id} (SF={sf}, d={distance_km:.1f}km)")

    def update_sensor_data(self, sensor_id, pressure, level, is_open):
        if sensor_id in self.sensors:
            self.sensors[sensor_id]['data'] = {'p': pressure, 'v': is_open, 'l': level}

    def get_packet_loss_rate(self):
        if self.total_transmissions == 0: return 0.0
        return (self.total_collisions / self.total_transmissions) * 100.0

    def step(self, current_time, timestep_s):
        received = []
        for s_id, s_node in self.sensors.items():
            # ONLY send if interval has passed (Realism Fix)
            if current_time - s_node['last_tx_time'] >= self.tx_interval_s:
                self.total_transmissions += 1
                if s_node['state'] == 0:
                    self.total_collisions += 1
                    if random.random() <= s_node['model'].p01: s_node['state'] = 1
                else:
                    received.append({'id': s_id, 'data': s_node['data']})
                    if random.random() <= s_node['model'].p10: s_node['state'] = 0
                s_node['last_tx_time'] = current_time
        return received

## 2. Water Network Management
This class handles the hydraulic network modifications, including adding IoT-enabled tanks and valves.

In [ ]:
class TankConfig:
    def __init__(self, tank_diameter, pipe_diameter, init_level, min_level, max_level):
        self.tank_diameter = tank_diameter
        self.pipe_diameter = pipe_diameter
        self.init_level = init_level
        self.min_level = min_level
        self.max_level = max_level

TANK_CONFIGS = {
    'Small':  TankConfig(tank_diameter=15.0, pipe_diameter=0.30, init_level=5.0,  min_level=0.5, max_level=10.0),
    'Medium': TankConfig(tank_diameter=25.0, pipe_diameter=0.40, init_level=8.0,  min_level=1.0, max_level=12.0),
    'Large':  TankConfig(tank_diameter=40.0, pipe_diameter=0.50, init_level=10.0, min_level=1.5, max_level=15.0)
}

class WaterNetworkManager:
    def __init__(self, wn_model):
        self.wn = mwntr.network.WaterNetworkModel(wn_model) if isinstance(wn_model, str) else wn_model
        self.iot_tanks = {}
        self.iot_valves = []
        
    def remove_existing_tanks(self):
        tanks = [name for name, node in self.wn.nodes() if node.node_type == 'Tank']
        for name in tanks:
            self.wn.remove_node(name, with_control=True)

    def add_iot_tanks(self, n_tanks: int = 3) -> List[str]:
        junctions = self.wn.junction_name_list
        target_nodes = random.sample(junctions, min(n_tanks, len(junctions)))
        tank_types = ['Small', 'Medium', 'Large']
        for junc_name, tank_type in zip(target_nodes, tank_types):
            junc_node = self.wn.get_node(junc_name)
            config = TANK_CONFIGS[tank_type]
            tank_name, valve_name = f"IoT_Tank_{tank_type}", f"IoT_Valve_{tank_type}"
            
            self.wn.add_tank(name=tank_name, elevation=junc_node.elevation + 40.0, init_level=config.init_level, min_level=config.min_level, max_level=config.max_level, diameter=config.tank_diameter, coordinates=(junc_node.coordinates[0] + 100, junc_node.coordinates[1] + 100))
            self.wn.add_pipe(name=valve_name, start_node_name=junc_name, end_node_name=tank_name, length=50.0, diameter=config.pipe_diameter, roughness=120, initial_status=LinkStatus.Closed)
            
            self.iot_tanks[tank_name] = {'type': tank_type, 'junction': junc_name, 'valve': valve_name}
            self.iot_valves.append(valve_name)
        return self.iot_valves

    def trigger_blackout(self, head_multiplier=0.4):
        """Simulates a crisis by drastically reducing reservoir pressure."""
        for res_name in self.wn.reservoir_name_list:
            res = self.wn.get_node(res_name)
            res.head_timeseries.base_value *= head_multiplier

    def set_simulation_options(self, timestep_s=300):
        self.wn.options.time.duration = timestep_s
        self.wn.options.time.hydraulic_timestep = timestep_s
        self.wn.options.time.report_timestep = timestep_s
        self.wn.options.hydraulic.demand_model = 'PDA'
        self.wn.options.hydraulic.minimum_pressure = 0.0
        self.wn.options.hydraulic.required_pressure = 20.0

## 4. Co-Simulation Engine (Optimized Core)
The orchestrator that synchronizes the hydraulic and communication domains step-by-step.

## 3. Intelligent Agent (Objective Function)
The agent optimizes the response to the crisis using the objective function:
$$ F(a) = (\alpha \cdot \Delta S) - (\beta \cdot T_{resp}) - (\gamma \cdot PL_f) $$
It monitors demand satisfaction and packet loss to decide when to deploy emergency water reserves.

In [ ]:
class CrisisManagementAgent:
    def __init__(self, water_net, lora_net):
        self.water_net, self.lora_net = water_net, lora_net
        self.alpha, self.beta, self.gamma = 1.0, 0.1, 0.5
        self.prev_s = 1.0
        self.crisis_start = None
        self.opened_count = 0

    def calculate_current_satisfaction(self, sim):
        """Calculates average demand satisfaction. Handles empty lists during initialization."""
        satisfied = []
        for j in sim._wn.junction_name_list:
            # Ensure the node exists in results and the list is not empty
            if j in sim.node_res['satisfied_demand'] and len(sim.node_res['satisfied_demand'][j]) > 0:
                satisfied.append(sim.node_res['satisfied_demand'][j][-1])
        
        return sum(satisfied) / len(satisfied) if satisfied else 1.0

    def compute_objective(self, s, pl, t):
        ds = s - self.prev_s
        tr = (t - self.crisis_start) / 3600.0 if self.crisis_start else 0
        fa = (self.alpha * ds) - (self.beta * tr) - (self.gamma * (pl/100.0))
        self.prev_s = s
        return fa

    def decide_action(self, step, t, s):
        if step == 20:
            self.crisis_start = t
            return "TRIGGER_CRISIS"
        
        if s < 0.92 and self.opened_count < 1: 
            self.opened_count = 1; return "OPEN_SMALL"
        if s < 0.85 and self.opened_count < 2:
            self.opened_count = 2; return "OPEN_MEDIUM"
        if s < 0.75 and self.opened_count < 3:
            self.opened_count = 3; return "OPEN_LARGE"
            
        return "NONE"

In [ ]:
class CoSimulationEngine:
    def __init__(self, network_file, duration_hours=24, step_min=5):
        self.timestep_s = step_min * 60
        self.n_steps = int((duration_hours * 3600) / self.timestep_s)
        self.water_net = WaterNetworkManager(network_file)
        self.water_net.remove_existing_tanks()
        self.water_net.add_iot_tanks(3)
        self.water_net.set_simulation_options(self.timestep_s)
        self.lora_net = LoRaSystem()
        for v in self.water_net.iot_valves: self.lora_net.register_sensor(v, random.uniform(1, 4))
        self.sim = MWNTRInteractiveSimulator(self.water_net.wn)
        self.agent = CrisisManagementAgent(self.water_net, self.lora_net)
        self.stats = {'time':[], 'satisfaction':[], 'packet_loss':[], 'tanks':[], 'reward':[]}

    def run_simulation(self):
        self.sim.init_simulation()
        t = 0.0
        print("Starting Realistic Co-Simulation...")
        for step in range(self.n_steps):
            t += self.timestep_s
            s = self.agent.calculate_current_satisfaction(self.sim)
            pl = self.lora_net.get_packet_loss_rate()
            fa = self.agent.compute_objective(s, pl, t)
            act = self.agent.decide_action(step, t, s)

            if act == "TRIGGER_CRISIS":
                self.water_net.trigger_blackout(0.25)
            elif "OPEN_" in act:
                # Open specific tank based on agent count
                target_valve = self.water_net.iot_valves[self.agent.opened_count - 1]
                self.sim.open_valve(target_valve)
                # Dynamic Frequency: sensors send more often during crisis!
                self.lora_net.tx_interval_s = 600 
                print(f"[T={t/3600:.1f}h] Agent opens {target_valve}. Emergency TX Frequency activated.")

            # Track Stats
            self.stats['time'].append(t); self.stats['satisfaction'].append(s*100)
            self.stats['packet_loss'].append(pl); self.stats['reward'].append(fa)
            self.stats['tanks'].append(self.agent.opened_count)
            
            self.sim.step_sim()
            self.lora_net.step(t, self.timestep_s)
        return self.sim.get_results()

## 5. Simulation

In [ ]:
# Verified absolute path of the network file
network_file = '/content/drive/MyDrive/Data Science/Honor_Program_private/progetto_locale/Dyn-WNTR/NET_4.inp'

import os
if os.path.exists(network_file):
    print(f" Network file found: {network_file}")
    # Simulation engine initialization (6 hours, 5 min steps)
    # Using step_min instead of timestep_minutes to match the updated class definition
    engine = CoSimulationEngine(network_file, duration_hours=6, step_min=5)
    results = engine.run_simulation()
    
    # Synthetic results visualization
    if results:
        print("\n Results Analysis:")
        print(f"- Pressures calculated for {len(results.node)} nodes")
        print(f"- Flow rates calculated for {len(results.link)} links")
else:
    print(f" Error: The file {network_file} is not accessible. Check Google Drive mount.")

In [ ]:
import matplotlib.pyplot as plt

if 'engine' in locals() and engine.stats['time']:
    print("Generating simulation analysis plots...")
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    time_hours = [t/3600 for t in engine.stats['time']]
    
    # Plot 1: Demand Satisfaction
    axes[0].plot(time_hours, engine.stats['satisfaction'], 'b-', linewidth=2, label='Satisfied Demand (%)')
    axes[0].axhline(y=85.0, color='r', linestyle='--', label='Agent Threshold (85%)')
    axes[0].set_ylabel('Satisfaction (%)')
    axes[0].set_title('Hydraulic Performance: Demand Satisfaction Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2: IoT Tanks Status
    # Fixed key from 'tanks_opened' to 'tanks'
    axes[1].step(time_hours, engine.stats['tanks'], 'g-', where='post', linewidth=2, label='Active Reservoirs')
    axes[1].set_ylabel('Number of Tanks')
    axes[1].set_ylim(-0.5, 3.5)
    axes[1].set_title('Cyber-Physical Response: Emergency Tank Activation')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Plot 3: Packet Loss and Objective Function
    ax2_twin = axes[2].twinx()
    axes[2].plot(time_hours, engine.stats['packet_loss'], 'orange', linewidth=2, label='Packet Loss (%)')
    ax2_twin.plot(time_hours, engine.stats['reward'], 'purple', linestyle=':', label='Objective F(a)')
    
    axes[2].set_xlabel('Time (hours)')
    axes[2].set_ylabel('Packet Loss (%)', color='orange')
    ax2_twin.set_ylabel('Objective Reward F(a)', color='purple')
    axes[2].set_title('Communication Quality and Agent Reward')
    
    lines1, labels1 = axes[2].get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    axes[2].legend(lines1 + labels2, labels1 + labels2, loc='upper right')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No simulation data found. Please run the simulation engine first.")